# Phase Exploration

In [3]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src" / "fall_detection").is_dir()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fall_detection.analysis.amplitude_pipeline import (
    AmplitudePacketReader,
)
from fall_detection.radio.preprocessing.phase import (
    HE20_VALID_SUBCARRIERS,
    decode_ltf_blocks,
)

# Add or remove IDs here; each session is loaded independently.
SESSION_IDS = ["20261104T081805521126Z", "20261109T131546367822Z"]
if not SESSION_IDS or len(set(SESSION_IDS)) != len(SESSION_IDS):
    raise ValueError("Select at least one distinct session ID.")
# Existing derived exports are used directly. Rebuild/verify them from the
# corresponding archive with the session exporter when source validation is needed.
from types import SimpleNamespace

sessions = {}
for session_id in SESSION_IDS:
    path = PROJECT_ROOT / "data/interim/parquet" / session_id
    required = [path / f"{name}.parquet" for name in ("attempts", "phases", "packets")]
    if not all(file.is_file() for file in required):
        raise FileNotFoundError(
            f"Missing derived Parquet tables for {session_id}: {path}"
        )
    sessions[session_id] = SimpleNamespace(
        path=path, attempts=pd.read_parquet(path / "attempts.parquet")
    )
attempts = pd.concat(
    [session.attempts for session in sessions.values()], ignore_index=True
)
display(
    attempts.groupby(["participant_id", "session_id"], dropna=False)
    .size()
    .rename("attempts")
    .reset_index()
)

,participant_id,session_id,attempts
0,akmal,20261109T131546367822Z,246
1,khaelano,20261104T081805521126Z,255


## Compare a shared scenario

Select one accepted attempt per session. The default is the first non-excluded attempt not marked bad; override attempt keys below. Times are relative to each action cue, not measured movement onset. Matching a scenario does not make movements or recording conditions identical.

In [4]:
SCENARIO_ID = "walk-to-fall__start_front__end_front"
# Optional overrides: {"session_id": "trials/.../attempt_01"}.
ATTEMPT_KEYS = {}
RECEIVER_IDS = ["rx-1", "rx-2", "rx-3", "rx-4"]
# Signed physical HE20 subcarrier indices; change these to inspect other tones.
SUBCARRIERS = [-100, -50, 20, 100]

selected = []
for session_id in SESSION_IDS:
    candidates = attempts.loc[
        attempts["session_id"].eq(session_id)
        & attempts["scenario_id"].eq(SCENARIO_ID)
        & attempts["status"].eq("accepted")
        & attempts["inclusion"].ne("exclude")
        & attempts["quality"].ne("bad")
    ].sort_values(["trial_ordinal", "attempt"])
    if session_id in ATTEMPT_KEYS:
        candidates = candidates.loc[
            candidates["attempt_key"].eq(ATTEMPT_KEYS[session_id])
        ]
    if candidates.empty:
        raise ValueError(f"No eligible attempt for {session_id} / {SCENARIO_ID}")
    selected.append(candidates.iloc[0])
selected_attempts = pd.DataFrame(selected).reset_index(drop=True)
display(
    selected_attempts[
        [
            "participant_id",
            "session_id",
            "scenario_id",
            "attempt_id",
            "attempt_key",
            "repetition",
            "quality",
        ]
    ]
)

,participant_id,session_id,scenario_id,attempt_id,attempt_key,repetition,quality
0,khaelano,20261104T081805521126Z,walk-to-fall__start_front__end_front,169/01,trials/169_walk-to-fall__start_front__end_fron...,1,unreviewed
1,akmal,20261109T131546367822Z,walk-to-fall__start_front__end_front,169/01,trials/169_walk-to-fall__start_front__end_fron...,1,unreviewed


In [5]:
recordings = {}
for row in selected_attempts.itertuples(index=False):
    session = sessions[row.session_id]
    phases = pd.read_parquet(session.path / "phases.parquet")
    action = phases.loc[
        phases["attempt_key"].eq(row.attempt_key)
        & phases["collection_phase"].eq("action")
    ]
    if len(action) != 1:
        raise ValueError(f"Expected one action cue for {row.attempt_key}")
    cue_ns = int(action.iloc[0]["start_ns"])
    reader = AmplitudePacketReader(session.path / "packets.parquet")
    try:
        for receiver_id in RECEIVER_IDS:
            frame = reader.read(row.attempt_key, receiver_id)
            frame = frame.filter(frame["is_sensing"])
            if frame.is_empty():
                raise ValueError(
                    f"No sensing packets for {row.attempt_key} / {receiver_id}"
                )
            if not set(SUBCARRIERS).issubset(HE20_VALID_SUBCARRIERS):
                raise ValueError("Select valid signed physical HE20 subcarriers.")
            raw = np.full((len(frame), len(SUBCARRIERS)), np.nan)
            calibrated = raw.copy()
            for i, packet in enumerate(frame.iter_rows(named=True)):
                if not packet["driver_valid"]:
                    continue
                values = np.asarray(packet["csi_i"], dtype=float) + 1j * np.asarray(
                    packet["csi_q"], dtype=float
                )
                if len(values) * 2 != packet["len"]:
                    raise ValueError("Invalid CSI payload length")
                blocks = decode_ltf_blocks(
                    values,
                    sig_mode=packet["sig_mode"],
                    bandwidth=packet["bandwidth"],
                    stbc=packet["stbc"],
                    first_word_invalid=bool(packet["first_word"]),
                )
                block = next((b for b in blocks if b.name == "HE-LTF"), None)
                if block is None:
                    raise ValueError("Unsupported CSI layout")
                lookup = {tone: j for j, tone in enumerate(block.subcarrier_indices)}
                for j, tone in enumerate(SUBCARRIERS):
                    index = lookup[tone]
                    raw[i, j] = block.raw_phases[index]
                    calibrated[i, j] = block.calibrated_phases[index]
            recordings[row.session_id, receiver_id] = {
                "participant": row.participant_id,
                "time": (frame["capture_time_ns"].to_numpy() - cue_ns) / 1e9,
                "raw": raw,
                "calibrated": calibrated,
            }
    finally:
        reader.close()

## Raw and calibrated phase

Each row shows a receiver and each column a signed physical subcarrier. Points are recorded packets, colored by session; no temporal filtering or resampling is applied. Invalid packets, null or guard slots, and amplitudes below 4 are masked. Raw phase is wrapped to ±π. Calibration unwraps each packet across frequency, removes a magnitude-weighted linear slope and offset, and wraps the residual; it is relative rather than an absolute phase reference.

In [6]:
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for representation in ("raw", "calibrated"):
    fig, axes = plt.subplots(
        len(RECEIVER_IDS),
        len(SUBCARRIERS),
        figsize=(4 * len(SUBCARRIERS), 2.6 * len(RECEIVER_IDS)),
        sharex=True,
        sharey=True,
        squeeze=False,
        layout="constrained",
    )
    for i, receiver_id in enumerate(RECEIVER_IDS):
        for j, tone in enumerate(SUBCARRIERS):
            ax = axes[i, j]
            for k, session_id in enumerate(SESSION_IDS):
                sample = recordings[session_id, receiver_id]
                ax.scatter(
                    sample["time"],
                    sample[representation][:, j],
                    s=2,
                    alpha=0.55,
                    color=colors[k % len(colors)],
                    label=f"{sample['participant']} ({session_id})",
                    rasterized=True,
                )
            ax.axvline(0, color="black", ls="--", lw=0.8)
            ax.set_ylim(-np.pi, np.pi)
            ax.set_yticks([-np.pi, 0, np.pi], ["−π", "0", "π"])
            ax.grid(alpha=0.2)
            if i == 0:
                ax.set_title(f"Subcarrier {tone:+d}")
            if j == 0:
                ax.set_ylabel(f"{receiver_id}\nPhase (rad)")
            if i == len(RECEIVER_IDS) - 1:
                ax.set_xlabel("Time from action cue (s)")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc="outside upper center", ncol=min(2, len(SESSION_IDS))
    )
    fig.suptitle(f"{representation.capitalize()} phase · {SCENARIO_ID}")
    output_path = (
        PROJECT_ROOT
        / "artifacts/notebooks/01_02_phase_exp/figures"
        / f"{representation}-phase-session-comparison.png"
    )
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=150)
    plt.show()
    print(f"Saved: {output_path}")

## Calibrated phase near zero

The same recorded points are shown with the vertical axis limited to ±0.5 rad by default. Values outside this range are clipped from view; change `PHASE_ZOOM_RADIANS` to adjust the range.

In [ ]:
PHASE_ZOOM_RADIANS = 0.5
if not 0 < PHASE_ZOOM_RADIANS <= np.pi:
    raise ValueError("Choose a phase zoom between 0 and π radians.")

fig, axes = plt.subplots(
    len(RECEIVER_IDS),
    len(SUBCARRIERS),
    figsize=(4 * len(SUBCARRIERS), 2.6 * len(RECEIVER_IDS)),
    sharex=True,
    sharey=True,
    squeeze=False,
    layout="constrained",
)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for i, receiver_id in enumerate(RECEIVER_IDS):
    for j, tone in enumerate(SUBCARRIERS):
        ax = axes[i, j]
        for k, session_id in enumerate(SESSION_IDS):
            sample = recordings[session_id, receiver_id]
            ax.scatter(
                sample["time"],
                sample["calibrated"][:, j],
                s=2,
                alpha=0.55,
                color=colors[k % len(colors)],
                label=f"{sample['participant']} ({session_id})",
                rasterized=True,
            )
        ax.axvline(0, color="black", ls="--", lw=0.8)
        ax.axhline(0, color="black", lw=0.5, alpha=0.4)
        ax.set_ylim(-PHASE_ZOOM_RADIANS, PHASE_ZOOM_RADIANS)
        ax.grid(alpha=0.2)
        if i == 0:
            ax.set_title(f"Subcarrier {tone:+d}")
        if j == 0:
            ax.set_ylabel(f"{receiver_id}\nPhase (rad)")
        if i == len(RECEIVER_IDS) - 1:
            ax.set_xlabel("Time from action cue (s)")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncol=min(2, len(SESSION_IDS)))
fig.suptitle(f"Calibrated phase near zero · {SCENARIO_ID}")
zoom_path = (
    PROJECT_ROOT
    / "artifacts/notebooks/01_02_phase_exp/figures/calibrated-phase-near-zero.png"
)
zoom_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(zoom_path, dpi=150)
plt.show()
print(f"Saved: {zoom_path}")

## Squared consecutive phase difference

For adjacent recorded sensing packets, compute the wrapped angular difference
$\Delta\phi_t = \operatorname{atan2}(\sin(\phi_t-\phi_{t-1}),\cos(\phi_t-\phi_{t-1}))$
and plot $(\Delta\phi_t)^2$ at the later packet's time. Wrapping avoids treating a crossing of ±π as a nearly $2\pi$ change. Missing or masked values mask adjacent differences. Packet intervals can vary or span missing packets, so this is not a rate; changes larger than π between packets cannot be distinguished from smaller wrapped changes. Calibration removes a packet-wise frequency trend, so raw and calibrated differences describe different representations.

In [ ]:
for sample in recordings.values():
    for representation in ("raw", "calibrated"):
        delta = np.diff(sample[representation], axis=0)
        sample[representation + "_squared_difference"] = (
            np.arctan2(np.sin(delta), np.cos(delta)) ** 2
        )

In [ ]:
for representation in ("raw", "calibrated"):
    fig, axes = plt.subplots(
        len(RECEIVER_IDS),
        len(SUBCARRIERS),
        figsize=(4 * len(SUBCARRIERS), 2.6 * len(RECEIVER_IDS)),
        sharex=True,
        sharey=True,
        squeeze=False,
        layout="constrained",
    )
    for i, receiver_id in enumerate(RECEIVER_IDS):
        for j, tone in enumerate(SUBCARRIERS):
            ax = axes[i, j]
            for k, session_id in enumerate(SESSION_IDS):
                sample = recordings[session_id, receiver_id]
                squared = sample[representation + "_squared_difference"][:, j]
                ax.scatter(
                    sample["time"][1:],
                    squared,
                    s=2,
                    alpha=0.55,
                    color=colors[k % len(colors)],
                    label=f"{sample['participant']} ({session_id})",
                    rasterized=True,
                )
            ax.axvline(0, color="black", ls="--", lw=0.8)
            ax.set_ylim(0, np.pi**2)
            ax.grid(alpha=0.2)
            if i == 0:
                ax.set_title(f"Subcarrier {tone:+d}")
            if j == 0:
                ax.set_ylabel(f"{receiver_id}\nSquared difference (rad²)")
            if i == len(RECEIVER_IDS) - 1:
                ax.set_xlabel("Time from action cue (s)")
    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, labels, loc="outside upper center", ncol=min(2, len(SESSION_IDS))
    )
    fig.suptitle(
        f"Squared consecutive {representation} phase difference · {SCENARIO_ID}"
    )
    path = (
        PROJECT_ROOT
        / "artifacts/notebooks/01_02_phase_exp/figures"
        / f"{representation}-phase-squared-difference.png"
    )
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150)
    plt.show()
    print(f"Saved: {path}")

## Calibrated squared differences near zero

This view uses the same wrapped squared differences with a vertical limit of 0.1 rad² by default. Larger values are clipped; adjust `SQUARED_PHASE_ZOOM_MAX` to change the range.

Lines connect consecutive available differences. Masked readings break the line, but unrecorded packets can leave time gaps that the line bridges; the line does not reconstruct missing values.

In [ ]:
SQUARED_PHASE_ZOOM_MAX = 0.1  # rad²
if not 0 < SQUARED_PHASE_ZOOM_MAX <= np.pi**2:
    raise ValueError("Choose a zoom limit greater than zero and at most π².")

fig, axes = plt.subplots(
    len(RECEIVER_IDS),
    len(SUBCARRIERS),
    figsize=(4 * len(SUBCARRIERS), 2.6 * len(RECEIVER_IDS)),
    sharex=True,
    sharey=True,
    squeeze=False,
    layout="constrained",
)
colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
for i, receiver_id in enumerate(RECEIVER_IDS):
    for j, tone in enumerate(SUBCARRIERS):
        ax = axes[i, j]
        for k, session_id in enumerate(SESSION_IDS):
            sample = recordings[session_id, receiver_id]
            squared = sample["calibrated_squared_difference"][:, j]
            ax.plot(
                sample["time"][1:],
                squared,
                lw=0.7,
                alpha=0.75,
                color=colors[k % len(colors)],
                label=f"{sample['participant']} ({session_id})",
                rasterized=True,
            )
        ax.axvline(0, color="black", ls="--", lw=0.8)
        ax.set_ylim(0, SQUARED_PHASE_ZOOM_MAX)
        ax.grid(alpha=0.2)
        if i == 0:
            ax.set_title(f"Subcarrier {tone:+d}")
        if j == 0:
            ax.set_ylabel(f"{receiver_id}\nSquared difference (rad²)")
        if i == len(RECEIVER_IDS) - 1:
            ax.set_xlabel("Time from action cue (s)")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncol=min(2, len(SESSION_IDS)))
fig.suptitle(f"Calibrated squared phase difference near zero · {SCENARIO_ID}", y=1.03)
zoom_path = (
    PROJECT_ROOT
    / "artifacts/notebooks/01_02_phase_exp/figures/calibrated-phase-squared-difference-zoom.png"
)
zoom_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(zoom_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {zoom_path}")

## Optional burst measurement definition

The function below defines an exploratory countdown-relative burst summary. Its threshold, gap limit, recovery window, and minimum support are experiment choices. It is available for subsequent comparisons; defining it does not compute or validate fall-detection results.

In [ ]:
"""Descriptive burst measurements on one receiver's phase-motion trace."""

from collections.abc import Mapping

import numpy as np


def summarize_phase_bursts(
    seconds: np.ndarray,
    scores: np.ndarray,
    intervals: Mapping[str, tuple[float, float]],
    *,
    threshold_quantile: float = 0.95,
    min_phase_scores: int = 20,
    max_gap_seconds: float = 0.05,
    recovery_window_seconds: float = 1.0,
) -> dict[str, float | int | bool]:
    """Summarize action bursts against the recording's countdown distribution.

    Args:
        seconds: Finite, nondecreasing packet times relative to attempt start.
        scores: Nonnegative circular changes in radians per lag. Missing scores
            and lag windows crossing phase boundaries must already be NaN.
        intervals: Exactly one positive countdown/action/recovery interval each,
            in seconds on the packet timeline, without overlap.
        threshold_quantile: Countdown quantile defining an above-baseline burst.
        min_phase_scores: Minimum finite countdown/action scores for usability.
            Recovery needs the same count within its initial observation window.
        max_gap_seconds: Largest supported inter-sample interval. Missing scores,
            duplicate times, and larger gaps break runs and duration support.
        recovery_window_seconds: Initial recovery window used for its median.

    Returns:
        Coverage, threshold, action strength, burst duration/count, rise time,
        and initial recovery measurements. Unusable measurements remain NaN.
        Durations sum only intervals with two finite endpoints; above-threshold
        duration requires both endpoints above threshold and is a lower bound.
        Rise time interpolates the first 10% and 90% crossings of excess over
        threshold in the run containing the action maximum. A missing preceding
        below-threshold sample makes that rise left-censored and unavailable.

    Raises:
        ValueError: Arrays, phase intervals, or parameters are invalid.
    """
    seconds = np.asarray(seconds, dtype=float)
    scores = np.asarray(scores, dtype=float)
    if seconds.ndim != 1 or scores.shape != seconds.shape:
        raise ValueError("Expected matching one-dimensional times and scores")
    if not np.isfinite(seconds).all() or np.any(np.diff(seconds) < 0):
        raise ValueError("Packet times must be finite and nondecreasing")
    if np.isinf(scores).any() or np.any(scores[np.isfinite(scores)] < 0):
        raise ValueError("Scores must be nonnegative or NaN")
    if not np.isfinite(threshold_quantile) or not 0 < threshold_quantile < 1:
        raise ValueError("threshold_quantile must lie between zero and one")
    if (
        not isinstance(min_phase_scores, int)
        or isinstance(min_phase_scores, bool)
        or min_phase_scores < 2
    ):
        raise ValueError("min_phase_scores must be an integer of at least two")
    if any(
        not np.isfinite(value) or value <= 0
        for value in (max_gap_seconds, recovery_window_seconds)
    ):
        raise ValueError("Gap and recovery window must be finite and positive")
    phases = ("countdown", "action", "recovery")
    if set(intervals) != set(phases):
        raise ValueError("Exactly countdown/action/recovery intervals are required")
    bounds = np.asarray([intervals[phase] for phase in phases], dtype=float)
    if (
        bounds.shape != (3, 2)
        or not np.isfinite(bounds).all()
        or np.any(bounds[:, 1] <= bounds[:, 0])
        or np.any(bounds[1:, 0] < bounds[:-1, 1])
    ):
        raise ValueError("Phase intervals must be positive, ordered, and disjoint")
    masks = {
        phase: (seconds >= start) & (seconds < stop)
        for phase, (start, stop) in zip(phases, bounds)
    }
    finite = np.isfinite(scores)
    result = {
        "usable": False,
        **{
            f"{phase}_valid_scores": int((mask & finite).sum())
            for phase, mask in masks.items()
        },
    }
    for name in (
        "baseline_median_rad",
        "threshold_rad",
        "action_p95_rad",
        "action_p95_excess_rad",
        "peak_excess_rad",
        "peak_delay_s",
        "action_supported_s",
        "action_supported_fraction",
        "above_threshold_s",
        "active_fraction",
        "burst_count",
        "longest_burst_s",
        "rise_10_90_s",
        "peak_burst_left_censored",
        "peak_burst_right_censored",
        "recovery_median_excess_rad",
    ):
        result[name] = np.nan
    recovery_start, recovery_stop = intervals["recovery"]
    recovery = (
        masks["recovery"]
        & finite
        & (seconds < min(recovery_stop, recovery_start + recovery_window_seconds))
    )
    result["recovery_window_valid_scores"] = int(recovery.sum())
    if result["countdown_valid_scores"] < min_phase_scores:
        return result
    baseline = scores[masks["countdown"] & finite]
    threshold = float(np.quantile(baseline, threshold_quantile))
    result["baseline_median_rad"] = float(np.median(baseline))
    result["threshold_rad"] = threshold
    if recovery.sum() >= min_phase_scores:
        result["recovery_median_excess_rad"] = float(
            np.median(scores[recovery]) - result["baseline_median_rad"]
        )
    if result["action_valid_scores"] < min_phase_scores:
        return result
    selected = np.flatnonzero(masks["action"])
    t, y = seconds[selected], scores[selected]
    valid = np.isfinite(y)
    dt = np.diff(t)
    connected = (dt > 0) & (dt <= max_gap_seconds)
    supported = connected & valid[:-1] & valid[1:]
    support = float(dt[supported].sum())
    result["action_supported_s"] = support
    result["action_supported_fraction"] = support / (
        intervals["action"][1] - intervals["action"][0]
    )
    if support == 0:
        return result
    result["usable"] = True
    result["action_p95_rad"] = float(np.nanquantile(y, 0.95))
    result["action_p95_excess_rad"] = result["action_p95_rad"] - threshold
    peak = int(np.nanargmax(y))
    result["peak_excess_rad"] = float(y[peak] - threshold)
    result["peak_delay_s"] = float(t[peak] - intervals["action"][0])
    above = valid & (y > threshold)
    active = supported & above[:-1] & above[1:]
    result["above_threshold_s"] = float(dt[active].sum())
    result["active_fraction"] = result["above_threshold_s"] / support
    starts = np.flatnonzero(above & ~np.r_[False, above[:-1] & connected])
    ends = np.flatnonzero(above & ~np.r_[above[1:] & connected, False])
    result["burst_count"] = len(starts)
    result["longest_burst_s"] = float(np.max(t[ends] - t[starts], initial=0))
    if not above[peak]:
        return result
    run = int(np.flatnonzero((starts <= peak) & (ends >= peak))[0])
    start, end = int(starts[run]), int(ends[run])
    left_censored = start == 0 or not (connected[start - 1] and valid[start - 1])
    right_censored = end == len(y) - 1 or not (connected[end] and valid[end + 1])
    result["peak_burst_left_censored"] = bool(left_censored)
    result["peak_burst_right_censored"] = bool(right_censored)
    if not left_censored:
        # Only interpolate scalar crossings between observed samples; no trace
        # resampling, smoothing, or bridging of missing packets is performed.
        rise_y, rise_t = y[start - 1 : peak + 1], t[start - 1 : peak + 1]
        crossings = []
        for fraction in (0.1, 0.9):
            level = threshold + fraction * (y[peak] - threshold)
            upper = int(np.flatnonzero(rise_y >= level)[0])
            lower = upper - 1
            weight = (level - rise_y[lower]) / (rise_y[upper] - rise_y[lower])
            crossings.append(rise_t[lower] + weight * (rise_t[upper] - rise_t[lower]))
        result["rise_10_90_s"] = float(crossings[1] - crossings[0])
    return result